<a href="https://colab.research.google.com/github/Srivani0369/Cloud-Based-File-Sharing/blob/main/resume2_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pdfminer.six python-docx docx2txt fuzzywuzzy[speedup] python-dateutil sentence-transformers scikit-learn pandas tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 28.5 MB/s eta 0:00:00


In [ ]:
from google.colab import files
import os

os.makedirs("resumes", exist_ok=True)
uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, f"resumes/{filename}")

print("Uploaded resumes:", os.listdir("resumes"))


In [ ]:
job_json = {
    "title": "Senior Machine Learning Engineer",
    "description": "Design ML models, deploy and monitor production pipelines.",
    "required_skills": ["Python", "TensorFlow", "PyTorch", "Machine Learning", "Deep Learning"],
    "preferred_skills": ["AWS", "Docker", "Kubernetes", "MLOps"],
    "min_years": 3,
    "required_degrees": ["Bachelor", "Master"]
}

import json
with open("job.json", "w") as f:
    json.dump(job_json, f, indent=4)

print("Job file created!")


In [ ]:
# --- FULL RESUME SCREENER CODE ---
# (Same code I provided earlier, but ready for Colab)


In [ ]:
!python ai_resume_screener.py --resumes_dir ./resumes --job_file job.json --out ranked.csv


In [ ]:
from google.colab import files
files.download("ranked.csv")


In [ ]:
# Colab-ready: Resume <-> Role matching + CSV export
# Paste this entire cell into Google Colab and run.

# 1) Install deps (first run may take ~1-2 minutes for model download)
!pip install --quiet PyPDF2 docx2txt sentence-transformers fuzzywuzzy[speedup] python-dateutil scikit-learn pandas tqdm

# 2) Imports
from google.colab import files
import os, re, io, json
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import PyPDF2
import docx2txt
from fuzzywuzzy import fuzz
from datetime import datetime

# 3) Define job roles (you can edit/add roles here)
job_roles = {
    "Data Scientist": """
    Looking for a Data Scientist skilled in Python, Machine Learning, Data Analysis,
    Pandas, NumPy, and visualization tools like Matplotlib or Seaborn.
    Experience with NLP, deep learning, and AI frameworks is a plus.
    """,

    "Web Developer": """
    We need a Web Developer proficient in HTML, CSS, JavaScript, React, and Node.js.
    Should have experience with database management (MySQL, MongoDB) and REST APIs.
    """,

    "AI Engineer": """
    Seeking an AI Engineer experienced in Python, Deep Learning, TensorFlow, Keras,
    Computer Vision, and Natural Language Processing. Must understand neural networks and model deployment.
    """,

    "Data Analyst": """
    Hiring a Data Analyst with knowledge of Excel, SQL, Power BI, and Python.
    Must be able to clean, analyze, and visualize data for business insights.
    """
}

# 4) Helper: text extraction from uploaded files
def extract_text_from_pdf_bytes(b):
    text = ""
    try:
        reader = PyPDF2.PdfReader(io.BytesIO(b))
        for p in reader.pages:
            page_text = p.extract_text()
            if page_text:
                text += page_text + "\n"
    except Exception as e:
        # fallback: ignore
        print("PDF parsing error:", e)
    return text

def extract_text_from_docx_bytes(b):
    # write to temp file and use docx2txt
    tmp = "/tmp/tmp_upload.docx"
    with open(tmp, "wb") as f:
        f.write(b)
    try:
        return docx2txt.process(tmp) or ""
    finally:
        try: os.remove(tmp)
        except: pass

def extract_text_from_txt_bytes(b):
    try:
        return b.decode('utf-8', errors='ignore')
    except:
        return str(b)

def extract_text_from_uploaded(filename, data_bytes):
    ext = filename.lower().split('.')[-1]
    if ext == "pdf":
        return extract_text_from_pdf_bytes(data_bytes)
    elif ext in ("docx","doc"):
        return extract_text_from_docx_bytes(data_bytes)
    elif ext in ("txt", "md"):
        return extract_text_from_txt_bytes(data_bytes)
    else:
        # try text decode
        return extract_text_from_txt_bytes(data_bytes)

# 5) Simple heuristics: extract keywords from role description
def keywords_from_role_text(text):
    # basic: split on commas/newlines and keep capitalized/technical tokens
    tokens = re.split(r'[\n,;]+', text)
    kws = []
    for t in tokens:
        t = t.strip()
        if not t: continue
        # keep multi-word tech names like "Machine Learning" by trimming
        if len(t.split()) <= 5 and re.search(r'[A-Za-z0-9\-+#]+', t):
            kws.append(t)
    # add split words too (JS libs)
    extras = []
    for k in kws:
        for part in re.split(r'\s+|/|-', k):
            if len(part) > 1:
                extras.append(part)
    kws = list(dict.fromkeys([k for k in (kws + extras) if len(k) > 1]))  # preserve order, dedupe
    return kws

# 6) Years-of-experience heuristic
def estimate_years_from_text(text):
    # look for explicit "X+ years" or "X years"
    m = re.search(r'(\d+(?:\.\d+)?)\s*\+\s*years', text, re.I)
    if not m:
        m = re.search(r'(\d+(?:\.\d+)?)\s+years', text, re.I)
    if m:
        try:
            return float(m.group(1))
        except:
            pass
    # look for date ranges like "Jan 2017 - Mar 2021" or "2018 - Present"
    ranges = re.findall(r'([A-Za-z]{3,9}\s*\d{4}|\d{4})\s*(?:-|\sto\s|–|—)\s*([A-Za-z]{3,9}\s*\d{4}|\d{4}|Present|present|Current|current)', text)
    total_years = 0.0
    for start, end in ranges:
        try:
            # parse start
            try:
                s = datetime.strptime(start.strip(), "%b %Y")
            except:
                try:
                    s = datetime.strptime(start.strip(), "%B %Y")
                except:
                    s = datetime.strptime(start.strip(), "%Y")
            # parse end
            if re.search(r'present|current', end, re.I):
                e = datetime.now()
            else:
                try:
                    e = datetime.strptime(end.strip(), "%b %Y")
                except:
                    try:
                        e = datetime.strptime(end.strip(), "%B %Y")
                    except:
                        e = datetime.strptime(end.strip(), "%Y")
            years = (e - s).days / 365.25
            if years > 0 and years < 50:
                total_years += years
        except:
            continue
    if total_years > 0:
        return round(total_years, 2)
    return 0.0

# 7) Ask user which role (or All)
print("Available Job Roles:")
for i, r in enumerate(job_roles.keys(), start=1):
    print(f"{i}. {r}")
print(f"{len(job_roles)+1}. All roles (score against every role)")

choice = input("\nEnter the number of the role to use (or press Enter to run All): ").strip()
if choice == "":
    selected_indices = list(range(1, len(job_roles)+2))
else:
    try:
        idx = int(choice)
        selected_indices = [idx]
    except:
        selected_indices = list(range(1, len(job_roles)+2))

selected_role_names = []
for i in selected_indices:
    if i <= len(job_roles):
        selected_role_names.append(list(job_roles.keys())[i-1])
    else:
        selected_role_names = list(job_roles.keys())
        break

print("\nSelected roles to score:", selected_role_names)

# 8) Upload resumes (you can upload multiple files)
print("\n📂 Upload resumes now (multiple selection allowed). Supported: PDF, DOCX, TXT")
uploaded = files.upload()
if not uploaded:
    raise SystemExit("No files uploaded. Re-run and upload resumes.")

# save uploaded to /content/resumes/ for convenience
resumes_dir = Path("resumes_colab")
resumes_dir.mkdir(exist_ok=True)
uploaded_files = []
for fname, blob in uploaded.items():
    path = resumes_dir / fname
    with open(path, "wb") as f:
        f.write(blob)
    uploaded_files.append(path)

print(f"\nSaved {len(uploaded_files)} files to {resumes_dir.resolve()}")

# 9) Prepare models and vectorizers
print("\nLoading sentence-transformer model (small, fast)...")
sbert = SentenceTransformer("all-MiniLM-L6-v2")  # small & fast
tfidf = TfidfVectorizer(stop_words='english')

# 10) Read resume texts
resume_texts = {}
for p in uploaded_files:
    with open(p, "rb") as f:
        b = f.read()
    txt = extract_text_from_uploaded(p.name, b)
    if not txt.strip():
        txt = "[no text extracted]"
    resume_texts[p.name] = txt

# 11) Build outputs: iterate roles & resumes -> compute scores
rows = []
for role_name in selected_role_names:
    role_text = job_roles[role_name].strip()
    role_kws = keywords_from_role_text(role_text)
    # prepare TF-IDF using role_text + all resumes for stable vectorization
    docs_for_vec = [role_text] + list(resume_texts.values())
    tfidf_matrix = tfidf.fit_transform(docs_for_vec)
    role_vec = tfidf_matrix[0:1]
    # precompute embedding for role
    role_emb = sbert.encode([role_text], convert_to_numpy=True)
    for i, (fname, rtext) in enumerate(resume_texts.items(), start=1):
        # TF-IDF similarity
        resume_vec = tfidf_matrix[i:i+1]
        tfidf_sim = float(cosine_similarity(role_vec, resume_vec)[0,0])
        # embedding similarity
        r_emb = sbert.encode([rtext], convert_to_numpy=True)
        emb_sim = float(cosine_similarity(role_emb, r_emb)[0,0])
        # normalize embedding sim from (-1,1) to (0,1)
        emb_sim_norm = (emb_sim + 1.0) / 2.0
        # combined score (weighted)
        combined = 0.55 * tfidf_sim + 0.45 * emb_sim_norm
        # simple skill matching (count keywords present)
        matched = []
        for kw in role_kws:
            if re.search(r'\b' + re.escape(kw) + r'\b', rtext, re.I):
                matched.append(kw)
            else:
                # fuzzy partial match
                if fuzz.partial_ratio(kw.lower(), rtext.lower()) >= 80:
                    matched.append(kw)
        # years estimate
        yrs = estimate_years_from_text(rtext)
        rows.append({
            "file": fname,
            "role": role_name,
            "tfidf_score": round(tfidf_sim*100, 2),
            "embed_score": round(emb_sim_norm*100, 2),
            "combined_score": round(combined*100, 2),
            "matched_keywords": "; ".join(matched),
            "estimated_years": yrs
        })

# 12) Save results as CSV and show top results
df = pd.DataFrame(rows).sort_values(['role','combined_score'], ascending=[True, False]).reset_index(drop=True)
out_csv = "resume_role_matching_results.csv"
df.to_csv(out_csv, index=False)
print(f"\n✅ Results saved to {out_csv} (one row per resume × role).")

# show top 10 rows
print("\nTop results (first 10 rows):")
display(df.head(10))

# 13) Provide download link
from google.colab import files as gfiles
print("\nYou can download the full CSV now:")
gfiles.download(out_csv)
